# Adversarial Payments Framework

**Mastercard Innovation Challenge 2026 — AI red teaming for payment security**

One framework, two attack surfaces, measured red vs blue.

We build a *constraint-aware* adversarial loop and run it against two completely
different targets — a tabular fraud detector and an LLM payment agent — and report the
same shape of number for both: **attack success before defense, attack success after
defense, and what the defense cost.**

| | Red team | Blue team | Headline metric |
|---|---|---|---|
| **Tabular surface** | Constraint-projected evasion search against an XGBoost detector | Adversarial retraining over 3 rounds | Attack Success Rate (ASR) |
| **Agentic surface** | Indirect prompt injection in memos, invoices, merchant names, dispute text | Injection classifier + tool scoping + HITL threshold | Exploit rate |

Both terminate in one table, `framework_scorecard`. Two rows is the entire claim that this
is a *framework* rather than two projects sharing a repo.

---

### How to read this notebook

**This notebook never trains anything.** It reads committed JSON from `artifacts/` and
renders it. That is deliberate (design spec §4.3): nothing heavy runs while a judge is
watching, and the results are visible even on a machine that cannot install XGBoost.

Every number below is pulled live from those artifact files at render time. Nothing is
typed into the prose. If an artifact is still seed data, this notebook prints **`TK`**
where the number would go and names the owner to chase — it will not show you a plausible
fake.

To recompute everything from scratch instead, see the last section.

## 0. Setup and provenance audit

Two switches matter here, and both default to *off* on purpose.

- **`RUN_ORCHESTRATED=0`** — the pipeline can run under Prefect 3, but the Day-1 gate
  (`scripts/check_prefect_offline.py`) showed that Prefect's "serverless" mode boots an
  ephemeral HTTP server on `127.0.0.1` and takes ~29 seconds to do it. That is fine on a
  laptop and a real risk on a locked-down judging machine or a kernel that blocks socket
  binding. So the notebook default is the plain-loop path, which executes identical tasks.
  Prefect still drives the dashboard's graph, where the 29s is paid once at build time.
- **`RECOMPUTE=0`** — read artifacts, do not retrain.

The cell below also runs a **provenance audit**. Every artifact carries a `placeholder`
flag; seed fixtures ship `true`. The audit refuses to let a `true` number reach the prose.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

# Stdlib only, on purpose: this cell must work on a machine where the project package
# was never pip-installed. The notebook is a reader, not a runtime.

RUN_ORCHESTRATED = os.getenv("RUN_ORCHESTRATED", "0").strip().lower() in {"1", "true", "yes", "on"}
RECOMPUTE        = os.getenv("RECOMPUTE",        "0").strip().lower() in {"1", "true", "yes", "on"}

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "artifacts").is_dir()),
    Path.cwd(),
)
ARTIFACTS = ROOT / "artifacts"

KINDS = {
    "detect_rounds":   ARTIFACTS / "detect"  / "rounds.json",
    "attack_rounds":   ARTIFACTS / "attack"  / "rounds.json",
    "attack_examples": ARTIFACTS / "attack"  / "examples.json",
    "agentic_redteam": ARTIFACTS / "agentic" / "redteam.json",
    "scorecard":       ARTIFACTS / "scorecard.json",
    "graph":           ARTIFACTS / "graph.json",
}

OWNER = {
    "detect_rounds":   "P1 (detector)",
    "attack_rounds":   "P2 (attack)",
    "attack_examples": "P2 (attack)",
    "agentic_redteam": "P3 (agentic)",
    "scorecard":       "P2 + P3",
    "graph":           "P2 (loop)",
}

ART: dict[str, dict] = {}
for kind, path in KINDS.items():
    try:
        ART[kind] = json.loads(path.read_text(encoding="utf-8"))
    except FileNotFoundError:
        ART[kind] = {"kind": kind, "placeholder": True, "payload": None, "_missing": True}


def is_real(kind: str) -> bool:
    """True only when this artifact holds numbers a real run produced."""
    a = ART.get(kind, {})
    return bool(a) and not a.get("placeholder", True) and not a.get("_missing")


def payload(kind: str):
    return ART.get(kind, {}).get("payload")


def tk(kind: str, note: str = "") -> str:
    """The visible admission that a number does not exist yet."""
    why = "file missing" if ART.get(kind, {}).get("_missing") else "still placeholder: true"
    return f"**TK** — {note or kind} ({why}; chase {OWNER.get(kind, '?')})"


print(f"repo root        : {ROOT}")
print(f"RUN_ORCHESTRATED : {int(RUN_ORCHESTRATED)}   (0 = plain loop, no Prefect server)")
print(f"RECOMPUTE        : {int(RECOMPUTE)}   (0 = read artifacts, never train)")
print()
print("PROVENANCE AUDIT")
print("-" * 64)
for kind in KINDS:
    a = ART[kind]
    if a.get("_missing"):
        state = "MISSING"
    elif a.get("placeholder", True):
        state = "PLACEHOLDER (seed data - not quotable)"
    else:
        state = f"REAL   git {a.get('git_sha', '?')}  {a.get('created_at', '?')[:19]}"
    print(f"  {kind:<17} {state}")
print("-" * 64)

_fake = [k for k in KINDS if not is_real(k)]
if _fake:
    print()
    print("!" * 64)
    print("!! NOT SUBMISSION-READY.")
    print(f"!! {len(_fake)} of {len(KINDS)} artifacts are seed data or missing:")
    for k in _fake:
        print(f"!!   - {k:<17} owner: {OWNER[k]}")
    print("!! Every figure below sourced from these renders TK instead of a number.")
    print("!" * 64)
else:
    print("\nAll artifacts real. Numbers below are from a genuine run.")

### Data provenance — stated up front, not buried

A reader is entitled to know what the numbers were computed *on* before they read them, and
to have that answer come from the pipeline rather than from our prose.

So it does. `scripts/fetch_data.py` writes `artifacts/data_provenance.json` recording where
the data actually came from, and the cell below reads that file and reports it. There is a
deterministic synthetic fallback in the loader for a locked-down environment where the
download fails — if it ever fires, the provenance file records `source: "synthetic"` and the
cell below prints a loud warning instead of a clean bill of health. **We cannot present
synthetic results as real ones here even by accident**, which is the point of wiring it this
way rather than writing a sentence and hoping it stays true.

In [ ]:
_prov_path = ARTIFACTS / "data_provenance.json"
try:
    PROV = json.loads(_prov_path.read_text(encoding="utf-8"))
except FileNotFoundError:
    PROV = None

if PROV is None:
    print("TK - dataset provenance: artifacts/data_provenance.json not found.")
    print("     Run scripts/fetch_data.py. No dataset claim is made until it exists.")
elif PROV.get("source") == "kaggle":
    print("DATASET: REAL")
    print(f"  Sparkov 'Credit Card Transactions Fraud Detection' "
          f"(Kaggle {PROV['kaggle_dataset']})")
    print(f"  {PROV['n_rows']:,} transactions from {PROV['n_cards']:,} cardholders")
    print(f"  {PROV['date_min'][:10]} to {PROV['date_max'][:10]}")
    print(f"  {PROV['n_fraud']:,} labelled frauds ({PROV['fraud_rate']:.3%} base rate)")
    print()
    print("  Downloaded anonymously via kagglehub - no Kaggle account or API token is")
    print("  required, so a judge with a network connection reproduces this exactly.")
    print()
    print(f"  At a {PROV['fraud_rate']:.3%} base rate, accuracy is meaningless and even")
    print("  ROC-AUC flatters. PR-AUC is the metric reported throughout.")
else:
    print("!" * 68)
    print("!! DATASET: SYNTHETIC - NOT THE REAL SPARKOV DATA")
    print(f"!! source    : {PROV.get('source')}")
    print(f"!! generator : {PROV.get('generator')}   seed: {PROV.get('seed')}")
    if PROV.get("warning"):
        print(f"!! warning   : {PROV['warning']}")
    print("!!")
    print("!! The Kaggle download failed and the deterministic fallback generator ran.")
    print("!! Every number below therefore describes Sparkov-SHAPED synthetic data, not")
    print("!! the real dataset. Do not quote these figures as real-dataset results.")
    print("!" * 68)

> **TK — LLM provenance.** P3 owns this line, and it is the one piece of provenance still
> resting on a human attestation rather than a file. The agentic numbers come from one of
> three regimes and it matters which: a **live model** (named), **cached real responses**
> replayed offline, or a **scripted stub** that never contacts a model. If it is a stub, the
> exploit-rate numbers describe our simulation of an agent rather than an agent — and this
> box will say exactly that.

## 1. Threat model

Everything downstream is a consequence of taking one question seriously: **what can an
attacker actually change?**

Most published adversarial-ML results answer "any input feature, by any amount". That is
the right answer for images, where every pixel is under the attacker's control. It is the
wrong answer for payments, and getting it wrong inflates every number you report.

A fraudster operating stolen card credentials inherits a great deal they cannot alter:

| The attacker **cannot** change | Because |
|---|---|
| The cardholder's age, gender, job, home city and its population | These are the victim's, not the attacker's. Stolen credentials come with them attached. |
| The issuer's historical view of the account | Prior transactions are already written; an attacker adds to that history, they do not rewrite it. |
| The clock, retroactively | A transaction is stamped when it happens. The attacker chooses *when to act* — which is a lever, see below — but cannot restamp a transaction afterwards. |

| The attacker **can** change | How |
|---|---|
| Amount | Enter a different number. Free. |
| Timing and pacing — hour, day, gap since last transaction, transactions per hour | Wait, or don't. Nearly free. |
| **Which merchant to hit** | Pick a different target — but see below, this is not a free single-feature move. |

And the piece that most published work drops: **choosing a merchant changes four features
at once.** Merchant category, terminal latitude, terminal longitude and the
cardholder-to-terminal distance move *together*, because they are four projections of one
decision. An attack that nudges `distance_km` while holding `merch_lat` fixed has produced
a transaction that could not physically occur.

That matters commercially, not just aesthetically: **an ASR measured over impossible
transactions is a number we would have to retract under questioning.** It reports the
detector as weaker than it is, against an adversary that cannot exist.

So the search space is constrained by three projections, applied at every step:

1. **Immutability** — frozen features are excluded from the search entirely.
2. **Feasibility** — mutable features are projected back into the plausible band observed
   in training (inner quantiles, not min/max, so one outlier cannot hand the attacker an
   enormous legal range), *and* coupled features move as a group or not at all.
3. **Sparsity** — minimise the L0 count of features touched. A five-feature evasion is a
   worse attack than a one-feature evasion even when both succeed, because it is harder to
   execute and easier to catch.

The same discipline transfers to the agentic surface. There, the "immutable" boundary is
which text a payment agent must ingest but can never trust: memos, invoice metadata,
merchant display names, chargeback dispute text. The attacker does not control the agent's
system prompt. They control the untrusted fields, and only those.

## 2. Why Sparkov, and specifically why not `creditcard.csv`

This is the most consequential decision in the project and it is worth two paragraphs,
because it is also where a lazier submission goes wrong.

The obvious dataset for a credit-card fraud demo is the ULB `creditcard.csv` — it is the
most-downloaded fraud dataset in existence, it is small, it is clean, and roughly every
fraud-detection tutorial on the internet uses it. We rejected it.

`creditcard.csv` is **PCA-anonymized**. Its features are `V1` through `V28`: 28 principal
components of an undisclosed original feature set, plus `Time` and `Amount`. There is no
merchant category. No terminal geography. No device. No cardholder demographics.

Now re-read §1. Our entire contribution is the claim that a payment-domain adversarial
attack must respect what an attacker can and cannot control. On `creditcard.csv` that
claim is **not merely hard to implement — it is undefined.** You cannot freeze the
merchant category, because there is no merchant category; it has been linearly mixed into
all 28 components. You cannot enforce that terminal geography moves as a coupled group,
because geography does not exist as an addressable quantity. Every projection in §1
degenerates to "perturb `V1`–`V28` freely", which is exactly the unconstrained attack we
are arguing against.

Worse, it degenerates *silently*. You can absolutely write a constraint-aware attack
engine, point it at `creditcard.csv`, and get a beautiful ASR-collapse curve out of it.
The number would be real and the claim attached to it would be a fiction — and a domain
judge who has actually worked with payments data would catch it in one question:
*"which of those V-columns is the MCC?"*

**Sparkov** (`kartik2112/fraud-detection`) keeps the raw columns, so the constraint story
is literally expressible in the data:

| Projection | What it binds | Underlying Sparkov columns |
|---|---|---|
| **Immutability** | Victim identity and home geography — excluded from the search | `age`, `gender`, `city_pop`, `lat`, `long`, `state`, `job` |
| **Coupling** | One merchant choice, four features moving together | `category` (MCC proxy), `merch_lat`, `merch_long`, and the derived `distance_km` |
| **Feasibility** | Plausible bands on the levers | `amt` within the account's historical range; timing within observed pacing |
| **Sparsity** | Cost of execution | L0 over the attackable set |

None of those three tiers is even *definable* over `V1`–`V28`. That is the argument in one
line.

The cost of this choice is honest and worth naming: Sparkov is itself **simulator-generated
data**, not a real payment network's traffic. Its fraud patterns were authored by a
generator, so a detector can learn them more cleanly than it would learn real fraud. We
accept that, because the alternative dataset makes our central claim inexpressible, and a
weaker dataset supporting a real claim beats a stronger dataset supporting a fake one.

The table below is not typed by hand — it is read from the frozen contract in
`src/adversarial_payments/schema.py`, which is the object the attack engine actually calls.
If the code and this narrative ever diverge, the cell fails loudly.

In [ ]:
import sys

_src = ROOT / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

try:
    from adversarial_payments import schema as S
    SCHEMA_OK = True
except Exception as exc:  # package not importable on this machine - degrade, don't crash
    SCHEMA_OK = False
    print(f"could not import schema.py ({exc.__class__.__name__}: {exc})")
    print("the projections table below is therefore unavailable; the argument above stands.")

if SCHEMA_OK:
    print(f"contracted features : {len(S.FEATURES)}")
    print(f"attackable surface  : {len(S.ATTACKABLE)} of {len(S.FEATURES)} "
          f"({len(S.ATTACKABLE) / len(S.FEATURES):.0%})")
    print()
    print("TIER 1  IMMUTABLE  - excluded from the search entirely")
    for c in sorted(S.FROZEN):
        print(f"    {c}")
    print()
    print("TIER 2  COUPLED    - move as a group or not at all")
    for i, g in enumerate(S.COUPLED_GROUPS, 1):
        print(f"    group {i}: {' + '.join(g)}")
    print()
    print("TIER 3  MUTABLE    - the attacker's direct levers")
    for c in sorted(S.MUTABLE):
        print(f"    {c}")

**A note on where the code and the design spec disagree.**

Design spec §3 sketched a two-tier split in which the immutability projection covered
merchant category, terminal geography and the timestamp. The frozen contract in `schema.py`
— which is what actually runs — implements the **three-tier** model of spec §4.2 instead:
merchant category and terminal geography are *coupled*, timing is *mutable*, and the
immutable tier is reserved for the victim's own attributes. §4.2 supersedes the §3 table.

The code is right and the §3 table was the earlier, looser sketch. An attacker genuinely does choose which
merchant to attack; that is a lever, not a constant. What they cannot do is choose it
*incoherently* — pick the category without the geography that comes with it. Modelling
merchant choice as coupled-but-attackable is both more honest and a strictly harder problem
for our defense, since it hands the red team a dimension the looser reading would have
frozen.

The same correction applies to time. `hour`, `day_of_week` and `is_night` are **mutable**,
not frozen, because an attacker genuinely chooses when to transact — and claiming otherwise
is exactly the sort of overreach a domain judge would push back on.

Note also that `merchant` and `trans_date_trans_time` are *raw columns*, not model features.
They feed the engineered features above; the model sees exactly the columns in
`schema.FEATURES`. We flag all of this rather than quietly letting two of our own documents
contradict each other.

## 3. The result — ASR collapse across adversarial rounds

The loop, unrolled over rounds *r* = 0, 1, 2:

```
train_detector(r) ─▶ score_detector(r) ─▶ PR-AUC_r, threshold_r
        ▲                    │
        │                    ▼
        │          generate_attacks(model_r)  ── under the three projections
        │                    │
        │                    ▼
        │          score_attacks(model_r) ─▶ ASR_r, mean L0, mean L2
        │                    │
        └── augment_trainset(train, adversarial_r)      [unroll edge]
```

Note the word **unrolled**. The red/blue process is a *cycle* — attack, detect, retrain,
attack again — and that co-evolution is the whole thesis. It becomes a DAG only once you
unroll it over rounds, so that round 1's retrained detector is a distinct node feeding
round 2's attacker. We say "unrolled loop", not "DAG", because a judge who knows the
difference will notice, and precision is free.

Two numbers have to move in opposite directions for this to mean anything:

- **ASR must fall** — the retrained detector resists the attack it was hardened against.
- **PR-AUC must hold** — the hardening did not cost meaningful accuracy on ordinary fraud.

A submission that shows only the first number has shown you a detector that learned to say
"fraud" more often. The pair is the claim.

In [ ]:
def _rows(kind):
    p = payload(kind)
    return p if isinstance(p, list) else []


HAVE_LOOP = is_real("attack_rounds") and is_real("detect_rounds")

if not HAVE_LOOP:
    print("TK - ASR / PR-AUC co-evolution table")
    for k in ("attack_rounds", "detect_rounds"):
        if not is_real(k):
            state = "missing" if ART[k].get("_missing") else "placeholder: true"
            print(f"     {k}: {state}  -> owner {OWNER[k]}")
    print()
    print("     Seed fixtures exist at these paths and contain plausible-looking numbers.")
    print("     They are NOT rendered here. A placeholder number that reaches a judge is")
    print("     worse than a visible gap, so this cell prints the gap.")
else:
    atk = {r["round"]: r for r in _rows("attack_rounds")}
    det = {r["round"]: r for r in _rows("detect_rounds")}
    rounds = sorted(set(atk) & set(det))

    hdr = f"{'round':>5} {'ASR':>8} {'PR-AUC':>8} {'mean L0':>8} {'mean L2':>8} {'queries':>8} {'adv added':>10}"
    print(hdr)
    print("-" * len(hdr))
    for r in rounds:
        a, d = atk[r], det[r]
        print(f"{r:>5} {a['asr']:>8.3f} {d['pr_auc']:>8.3f} {a['mean_l0']:>8.2f} "
              f"{a['mean_l2']:>8.2f} {a['median_queries']:>8d} {d['n_adversarial_added']:>10,}")
    print()

    r0, rN = rounds[0], rounds[-1]
    asr0, asrN = atk[r0]["asr"], atk[rN]["asr"]
    pr0, prN   = det[r0]["pr_auc"], det[rN]["pr_auc"]
    drop = (asr0 - asrN) / asr0 if asr0 else float("nan")

    print(f"ASR      round {r0} -> {rN}:  {asr0:.3f} -> {asrN:.3f}   "
          f"({drop:.1%} relative REDUCTION - red team loses ground)")
    print(f"PR-AUC   round {r0} -> {rN}:  {pr0:.3f} -> {prN:.3f}   "
          f"({(prN - pr0) / pr0:+.1%} relative - the cost of hardening)")
    print(f"mean L0  round {r0} -> {rN}:  {atk[r0]['mean_l0']:.2f} -> {atk[rN]['mean_l0']:.2f}"
          "   (features an attacker must touch to still succeed)")

In [ ]:
try:
    import matplotlib.pyplot as plt
    PLOT_OK = True
except Exception:
    PLOT_OK = False
    print("matplotlib unavailable - the table above carries the same information.")

if PLOT_OK and HAVE_LOOP:
    rs   = rounds
    asrs = [atk[r]["asr"] for r in rs]
    prs  = [det[r]["pr_auc"] for r in rs]
    l0s  = [atk[r]["mean_l0"] for r in rs]

    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

    ax.plot(rs, asrs, marker="o", lw=2.2, color="#c0392b", label="Attack Success Rate (red)")
    ax.plot(rs, prs,  marker="s", lw=2.2, color="#2471a3", label="PR-AUC (blue)")
    ax.set_xlabel("adversarial round")
    ax.set_ylim(0, 1)
    ax.set_xticks(rs)
    ax.grid(alpha=0.25)
    ax.legend(loc="center right", fontsize=9)
    ax.set_title("Red falls, blue holds", fontsize=11)

    ax2.bar([str(r) for r in rs], l0s, color="#7d3c98")
    ax2.set_xlabel("adversarial round")
    ax2.set_ylabel("mean L0 (features touched)")
    ax2.grid(alpha=0.25, axis="y")
    ax2.set_title("Cost to the attacker of still succeeding", fontsize=11)

    fig.tight_layout()
    plt.show()
elif PLOT_OK:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.text(0.5, 0.5, "TK\nASR co-evolution\nawaiting real artifacts from P1 + P2",
            ha="center", va="center", fontsize=13, color="#c0392b")
    ax.set_axis_off()
    plt.show()

### What this chart does *not* show — the caveats before a judge finds them

Four, and we would rather say them than be asked them.

1. **The attacker has white-box query access to a fixed model.** The search calls
   `predict_proba` as often as it needs. A real fraudster probing a production issuer gets
   a binary approve/decline, heavy rate limiting, and a model that changes underneath them.
   Our ASR is therefore an **upper bound on a strong attacker**, not a forecast of live
   losses. That framing is the useful one for a red team — you want the pessimistic bound —
   but it is not a loss estimate and we do not present it as one.

2. **The round-*r*+1 detector was trained on round-*r*'s adversarial examples.** So the ASR
   collapse partly measures *this specific attack being memorised*, not general robustness.
   Honest adversarial-training results always carry this caveat and most write-ups omit it.
   The meaningful generalisation test is a held-out attack the detector never saw in
   training — see the roadmap section.

3. **Mean L0 rising across rounds is the real story, and it is a weaker story than "the
   attack stopped working."** The attack does not become impossible; it becomes
   *expensive*. Each additional feature an attacker must touch is another behaviour to
   coordinate and another signal for downstream monitoring. Defense-in-depth economics, not
   a solved problem.

4. **Sparkov is simulator-generated.** See §2. A detector trained on authored fraud
   patterns learns them more cleanly than it would learn real fraud, so absolute PR-AUC
   here should be read as *relative* across rounds, never as a number to expect in
   production.

### A worked example — what a successful evasion actually looks like

Aggregate rates are easy to nod along to. This is one transaction, before and after.

In [ ]:
if not is_real("attack_examples"):
    print("TK - worked evasion example  (owner: " + OWNER["attack_examples"] + ")")
    print("     A single before/after transaction is the most persuasive artifact we have;")
    print("     it is left blank rather than illustrated with seed data.")
else:
    for ex in _rows("attack_examples"):
        print(f"transaction {ex['id']}   round {ex['round']}")
        print(f"  detector p(fraud): {ex['orig_prob']:.3f}  ->  {ex['adv_prob']:.3f}")
        print(f"  features touched (L0 = {len(ex['touched'])}):")
        for d in ex["touched"]:
            print(f"      {d['feature']:<26} {d['before']:>12,.2f}  ->  {d['after']:>12,.2f}")
        if SCHEMA_OK:
            touched = {d["feature"] for d in ex["touched"]}
            illegal = touched & S.FROZEN
            print(f"  immutability check: {'VIOLATED ' + str(sorted(illegal)) if illegal else 'clean - no frozen feature moved'}")
            for g in S.COUPLED_GROUPS:
                hit = touched & set(g)
                if hit and hit != set(g):
                    print(f"  coupling check: PARTIAL group move {sorted(hit)} of {list(g)}"
                          " - this transaction may not be physically realisable")
        print()

## 4. The second surface — agentic exploit rate, before and after

Same framework, an entirely different modality. This is the section that decides whether we
built *a framework* or *a project*.

A mock payment agent is given three real tools — `check_balance`, `initiate_transfer`,
`update_payee` — and then fed untrusted text in the four places a payment system genuinely
has to ingest it:

- **transaction memos** — free text the payer writes
- **invoice metadata** — supplied by the biller
- **merchant display names** — supplied by the merchant
- **chargeback dispute text** — supplied by the disputing party

These are the agentic analogue of §1's threat model. The attacker does not control the
system prompt any more than a carder controls the victim's age. They control the untrusted
fields, and an indirect prompt injection is the exploit that turns text the agent must read
into instructions the agent obeys.

Defenses, in increasing order of how much they cost to operate:

1. **Injection classifier** on every untrusted field before it enters context.
2. **Tool scoping** — the agent's ability to call `update_payee` is bound to the
   conversation's authenticated intent, not to whatever it just read.
3. **HITL threshold** — transfers above a value threshold require a human, unconditionally.

Scored against **OWASP LLM Top 10** and **MITRE ATLAS AML.T0051** so the categories mean
something outside this repo.

In [ ]:
if not is_real("agentic_redteam"):
    print("TK - agentic exploit rate by category  (owner: " + OWNER["agentic_redteam"] + ")")
    print()
    print("     Also outstanding and required before this section can be quoted:")
    print("       - whether these came from a live model, cached real responses, or a stub")
    print("       - whether the defense-cost latency was measured or estimated")
else:
    rows = _rows("agentic_redteam")
    hdr = f"{'category':<34} {'OWASP':<7} {'n':>5} {'before':>9} {'after':>9} {'delta':>8}"
    print(hdr)
    print("-" * len(hdr))
    tot_n = tot_b = tot_a = 0
    for r in rows:
        n, b, a = r["attempts"], r["success_before"], r["success_after"]
        tot_n, tot_b, tot_a = tot_n + n, tot_b + b, tot_a + a
        rb, ra = b / n, a / n
        print(f"{r['category']:<34} {r['owasp_id']:<7} {n:>5} {rb:>8.1%} {ra:>8.1%} {ra - rb:>+8.1%}")
    print("-" * len(hdr))
    print(f"{'OVERALL':<34} {'':<7} {tot_n:>5} {tot_b / tot_n:>8.1%} {tot_a / tot_n:>8.1%} "
          f"{(tot_a - tot_b) / tot_n:>+8.1%}")
    print()
    print("A representative injection from each category:")
    for r in rows:
        print(f"  [{r['owasp_id']}] {r['category']}")
        print(f"        {r['example_injection']}")

**Caveats, same standard as §3.**

- **A residual exploit rate above zero is the honest result and we are not going to round it
  to zero.** Prompt injection is not solved. Anyone claiming a 100% block rate on a defense
  layer this cheap is measuring their own test set.
- **The red team and the defense classifier may share a family of models,** which flatters
  the classifier. Injections written by the same model family the classifier was tuned on
  are easier to catch than a determined human's.
- **Our injection corpus is finite and authored by us.** It is a floor on the attack
  surface, not a census of it.
- **The HITL threshold defense is not free and its cost is not a latency number.** It
  transfers work to a human reviewer. On a real payment book that is an operational cost
  measured in headcount, and a scorecard that only counts milliseconds is hiding it.

## 5. `framework_scorecard` — the terminal node

Two rows. This table is the argument.

If red-teaming a gradient-boosted tabular classifier and red-teaming an LLM agent with tool
access produce results in the *same shape* — attack success before, attack success after,
cost of the defense — then what we built is a method that transfers across modalities, and
the two surfaces are evidence for it rather than two demos in two tabs.

The obvious next rows, which the research report describes and we deliberately did not build
(they are 6–12 engineer-months, and two working surfaces already carry the claim): voice
anti-spoofing, and graph/AML topology. They belong on the roadmap, cited, not in this table
pretending to be measured.

In [ ]:
if not is_real("scorecard"):
    print("TK - framework_scorecard  (owner: " + OWNER["scorecard"] + ")")
    print("     Blocked on the tabular row (P2) and the agentic row (P3).")
    print("     This is the single most important table in the submission.")
else:
    rows = _rows("scorecard")
    w = max(len(r["surface"]) for r in rows) + 2
    hdr = f"{'surface':<{w}} {'metric':<22} {'before':>9} {'after':>9} {'reduction':>10}  defense cost"
    print(hdr)
    print("-" * (len(hdr) + 20))
    for r in rows:
        b, a = r["attack_success_before"], r["attack_success_after"]
        red = (b - a) / b if b else float("nan")
        print(f"{r['surface']:<{w}} {r['primary_metric']:<22} {b:>8.1%} {a:>8.1%} "
              f"{red:>9.0%}  {r['defense_cost']}")
    print()
    print("Both surfaces, one shape of result. That is the framework claim, stated as a table.")

## 6. Reproducing this

**In 30 seconds, with no Python at all** — open the static dashboard export. It is a
pre-built HTML bundle with the artifact JSON inlined at build time; there is no server, no
backend and nothing to install. Same numbers as this notebook, same source files.

**This notebook, as-is** — needs only the standard library plus `matplotlib` for the two
charts. It reads `artifacts/` and trains nothing, by design (spec §4.3): the demo cannot
fail mid-presentation because nothing heavy runs during it, and a judge whose machine
cannot build XGBoost still sees every result.

**Actually recomputing the numbers**, for anyone who wants to verify they are real rather
than take our word for it:

```bash
uv venv --python 3.12                  # 3.14 has no wheels for this ML stack yet
uv pip install -e ".[dev]"
python scripts/fetch_data.py           # Sparkov, ~350 MB, needs Kaggle credentials
RECOMPUTE=1 python -m adversarial_payments.loop.flows
```

Then re-run this notebook. Every figure above re-reads the regenerated JSON. Nothing is
hardcoded in the prose, so if your run disagrees with ours, this notebook will show *your*
numbers and the disagreement will be visible rather than buried.

| Env var | Default | Effect |
|---|---|---|
| `RECOMPUTE` | `0` | `1` retrains and re-attacks from scratch |
| `RUN_ORCHESTRATED` | `0` **in this notebook** | `1` runs the same tasks under Prefect 3 — boots an ephemeral local server, ~29s |
| `LLM_LIVE` | `0` | `1` calls a live model; `0` replays cached responses with zero network |
| `SAMPLE_ROWS` | full | Row cap for fast iteration |

### Why `RUN_ORCHESTRATED` defaults to 0 here

The Day-1 gate did pass — `scripts/check_prefect_offline.py` completes with no remote API.
But the log shows what "serverless" means in Prefect 3:

```
Starting temporary server on http://127.0.0.1:8684
... Finished in state Completed()        <- 29 seconds later
```

It binds a port. On a locked-down judging machine, or a kernel that blocks socket binding,
that is a demo that hangs for half a minute and then may not recover. The plain-loop path
executes identical tasks with no server, so it is the notebook default and Prefect stays
where its 29 seconds are paid once, at dashboard build time.

### Roadmap — named, not claimed

Deliberately out of scope, and we would rather list them than imply they exist: transfer
tests against attacks the detector never saw in training (the real generalisation
question raised in §3), voice anti-spoofing, graph/AML topology detection, streaming
inference under a p99 latency budget, and federated training with differential privacy.
Each is described in the background research and none of them is in this notebook.